# Two phase oscillators with a tabulated coupling

Two oscillators with frequencies `w1`, `w2` interact through a function `h` given as a
table of 101 samples (`table h % 101 0 2π ...`). The phase difference locks when the
frequency difference is small compared with the coupling `a`.

*Based on an example from XPP Studio (Peter Ashwin and Bard Ermentrout, GPL-3.0). Every number below is computed by XPPAUT 8.1 and its AUTO through pyauto-xpp.*

In [1]:
import plotly.graph_objects as go
from pyxpp.builder import ModelBuilder
from xpptut import Explorer, load, orbit_diagram, show_figure

## The model

Written with `ModelBuilder`, statement by statement as in an `.ode` file. Edit it and rerun the cells below.

In [2]:
mb = (ModelBuilder("phase_oscillators_table")
      .table("h", n=101, lo=0, hi=6.283185, expr="sin(t)+.6+.4*cos(t)+.2*sin(2*t)")
      .ode("x1", "w1+a*h(mod(x2-x1,6.283185))")
      .ode("x2", "w2+a*h(mod(x1-x2,6.283185))")
      .aux("phi", "mod(x2-x1,6.283185)")
      .parameter(w1=1, w2=1.1, a=0.15)
      .initial(x1=0, x2=2)
      .option(fold="x1", tor_per=6.283185)
      .option(fold="x2")
      .option(total=200, dt=0.05))
print(mb.render())

table h % 101 0 6.283185 sin(t)+.6+.4*cos(t)+.2*sin(2*t)
x1'=w1+a*h(mod(x2-x1,6.283185))
x2'=w2+a*h(mod(x1-x2,6.283185))
aux phi=mod(x2-x1,6.283185)
par w1=1,w2=1.1,a=0.15
init x1=0,x2=2
@ fold=x1,tor_per=6.283185
@ fold=x2
@ total=200,dt=0.05
done



## Explore

The sliders, the direction-field menu and the AUTO panel all call XPP. `ex.m` is the pyxpp model, for anything else.

In [3]:
ex = Explorer(mb, kind="plane", x="x1", y="x2", window=(0, 6.2832, 0, 6.2832), df=False, nullclines=False,
              fold={"x1": 6.283185, "x2": 6.283185}, sliders={"w2": (0.6, 1.4), "a": (0, 0.5)}, show=["phi"],
              title="Phase oscillators")
ex

(run this cell with the kernel to get the interactive explorer)

## Going further

**Locking range.** The phase difference settles (locks) only while `w2 - w1` is inside
the range set by the odd part of `h`.

In [4]:
import numpy as np
W2 = np.linspace(0.6, 1.4, 60)
spread = [np.ptp(ex.m.run(keep=False, w2=w, total=400, trans=200)["phi"]) for w in W2]
show_figure(go.Figure(go.Scatter(x=W2, y=spread, mode="lines+markers")), xaxis_title="w2",
            yaxis_title="range of phi (0 = locked)", height=300)

## Things to try

- Set `a` to 0: no locking at all; increase it and watch the locked range grow.